# 03 Robustness Evaluation

## Real-World Clinical Degradation Simulation

The original dataset represents an idealised, clean clinical environment. In practice, maternal health monitoring systems face:

- **Sensor measurement errors**: Blood pressure devices may report values that deviate from the true reading due to calibration drift, device quality, or environmental factors.
- **Missing clinical measurements**: Incomplete records arise from workflow issues, data entry failures, or temporary device faults.
- **Resource-constrained environments**: Equipment-specific absences (e.g., blood glucose test strips unavailable) cause structured, non-random data loss.

To evaluate model reliability under realistic deployment scenarios, three severity tiers of controlled degradation are applied to the test data. These tiers represent a spectrum from minor measurement imperfections to severe resource limitations.

**Degradation levels were selected as controlled experimental stress-test scenarios, not as exact estimates from a specific clinical environment.**

| Tier | BP Drift | Missing Rate | Temp Bias |
|------|----------|--------------|-----------|
| Pristine Baseline | 0 mmHg | 0% | 0 |
| Tier 1 (Mild) | ±5 mmHg | 5% | -0.1 to -0.5°F |
| Tier 2 (Moderate) | ±10 mmHg | 15% | -0.3 to -1.0°F |
| Tier 3 (Severe) | ±15 mmHg | 30% | -0.5 to -1.5°F |

**Research Question:** Under increasing levels of missing clinical measurements and sensor noise, which classical ML classifier remains the most reliable?

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.metrics import confusion_matrix
from IPython.display import display
import warnings
warnings.filterwarnings('ignore')

sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath('__file__'))))

from src.preprocessing import load_dataset
from src.models import get_models
from src.config import N_SPLITS, RANDOM_STATE, TIERS, DEGRADATION_SEEDS, HIGH_RISK_RECALL_DROP_THRESHOLD
from src.degradation import DataDegradationEngine
from src.metrics import calculate_metrics

# Output directories (relative to notebooks/ -> ../results/)
os.makedirs("../results/csv", exist_ok=True)
os.makedirs("../results/tables", exist_ok=True)
os.makedirs("../results/figures", exist_ok=True)

X, y, feature_names, lb, _ = load_dataset()
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

print(f"Dataset loaded: {X.shape[0]} records, {X.shape[1]} features")
print(f"Folds: {N_SPLITS} | Seeds: {len(DEGRADATION_SEEDS)} | Tiers: {len(TIERS)}")


In [ ]:
cv_results = []
severe_cms = {}   # Accumulate confusion matrices for Tier 3 only

print("STARTING ROBUSTNESS EVALUATION MATRIX...")
print(f"Total evaluations: {N_SPLITS} folds x {len(DEGRADATION_SEEDS)} seeds x {len(TIERS)} tiers x n_models")
print()

for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):
    print(f"  Fold {fold + 1}/{N_SPLITS}...")

    X_train = pd.DataFrame(X[train_idx], columns=feature_names)
    X_test  = pd.DataFrame(X[test_idx],  columns=feature_names)
    y_train, y_test = y[train_idx], y[test_idx]
    y_test_bin = lb.transform(y_test)

    # Train fresh models on clean training data
    models = get_models(random_state=RANDOM_STATE)
    for name, model in models.items():
        model.fit(X_train, y_train)
        if name not in severe_cms:
            severe_cms[name] = np.zeros((3, 3), dtype=int)

    # MICE fitted ONLY on clean training data — never on corrupted test data
    mice = IterativeImputer(random_state=RANDOM_STATE, max_iter=10)
    mice.fit(X_train)

    for seed in DEGRADATION_SEEDS:
        engine = DataDegradationEngine(random_state=seed)

        for tier_name, tier_cfg in TIERS.items():

            # Apply all degradations for this tier via the unified method
            X_corrupted = engine.apply_degradation(X_test, tier_cfg)

            # Validate actual missing rate
            validation = engine.calculate_missing_rate(X_test, X_corrupted)
            actual_missing = validation["overall_actual_rate"]

            # Impute missing values using training-fitted MICE
            X_defended = pd.DataFrame(
                mice.transform(X_corrupted), columns=feature_names)

            for name, model in models.items():
                y_pred = model.predict(X_defended)
                y_prob = model.predict_proba(X_defended)

                acc, macro_f1, high_risk_recall, brier = calculate_metrics(
                    y_test, y_pred, y_prob, y_test_bin)

                cv_results.append({
                    "Fold":                fold + 1,
                    "Seed":                seed,
                    "Tier":                tier_name,
                    "Classifier":          name,
                    "BP_Noise_mmHg":       tier_cfg["bp_drift"],
                    "Missing_Rate_Target": tier_cfg["missing_rate"],
                    "Actual_Missing_Rate": actual_missing,
                    "Temp_Bias_Low":       tier_cfg["temp_bias"][0],
                    "Temp_Bias_High":      tier_cfg["temp_bias"][1],
                    "Accuracy":            acc,
                    "Macro F1":            macro_f1,
                    "High Risk Recall":    high_risk_recall,
                    "Brier Score":         brier
                })

                # Accumulate confusion matrix for Tier 3 only (across all seeds)
                if tier_name == "Tier 3 (Severe)":
                    severe_cms[name] += confusion_matrix(y_test, y_pred, labels=[0, 1, 2])

print()
print("✅ EVALUATION COMPLETE.")


In [ ]:
cv_df = pd.DataFrame(cv_results)

# Save raw results
cv_df.to_csv("../results/csv/robustness_results.csv", index=False)
print(f"✅ CSV saved  ({len(cv_df)} rows) -> results/csv/robustness_results.csv")


In [ ]:
# --- Summary Table ---
# Mean performance across all folds and seeds, grouped by Classifier + Tier
metrics_cols = ["Accuracy", "Macro F1", "High Risk Recall", "Brier Score"]

summary_df = (
    cv_df
    .groupby(["Classifier", "Tier"])[metrics_cols]
    .agg(["mean", "std"])
)
summary_df.to_csv("../results/tables/robustness_summary.csv")
print("✅ Tables saved -> results/tables/robustness_summary.csv")

# --- Noise Tracking Summary ---
degradation_summary = (
    cv_df[["Tier", "BP_Noise_mmHg", "Missing_Rate_Target",
           "Actual_Missing_Rate", "Temp_Bias_Low", "Temp_Bias_High"]]
    .drop_duplicates(subset=["Tier"])
    .set_index("Tier")
)
degradation_summary.to_csv("../results/tables/degradation_summary.csv")

print()
print("Degradation Summary:")
display(degradation_summary)


In [ ]:
# --- Baseline Comparison and Performance Drop Analysis ---
mean_df = cv_df.groupby(["Classifier", "Tier"])[metrics_cols].mean()

clean_df = mean_df.xs("Pristine Baseline", level="Tier")

drop_rows = []
for clf in mean_df.index.get_level_values("Classifier").unique():
    for tier in [t for t in TIERS.keys() if t != "Pristine Baseline"]:
        try:
            clean_recall = clean_df.loc[clf, "High Risk Recall"]
            tier_recall  = mean_df.loc[(clf, tier), "High Risk Recall"]
            drop = clean_recall - tier_recall
            flagged = drop > HIGH_RISK_RECALL_DROP_THRESHOLD
            drop_rows.append({
                "Classifier":       clf,
                "Tier":             tier,
                "Clean Recall":     round(clean_recall, 4),
                "Degraded Recall":  round(tier_recall, 4),
                "Recall Drop":      round(drop, 4),
                "Unsafe (>10%)":    flagged
            })
        except KeyError:
            pass

drop_df = pd.DataFrame(drop_rows)
drop_df.to_csv("../results/tables/model_comparison.csv", index=False)

print("Performance Drop (High Risk Recall, Clean vs Degraded):")
display(drop_df)
print()
print(f"Safety criterion: flag if recall drop > {HIGH_RISK_RECALL_DROP_THRESHOLD:.0%}")
print("(This is an experimental research threshold, not a clinical guideline.)")


In [ ]:
# --- Degradation Curve Figures ---
tier_order = list(TIERS.keys())
cv_df["Tier"] = pd.Categorical(cv_df["Tier"], categories=tier_order, ordered=True)

# Helper
def save_degradation_curve(metric, filename, lower_better=False):
    plt.figure(figsize=(10, 6))
    sns.lineplot(data=cv_df, x="Tier", y=metric, hue="Classifier",
                 marker='o', errorbar='sd')
    note = " (Lower is Better)" if lower_better else " (Higher is Better)"
    plt.title(f"{metric} vs Degradation Severity")
    plt.ylabel(metric + note)
    plt.xlabel("Degradation Tier")
    plt.xticks(rotation=15)
    plt.tight_layout()
    plt.savefig(f"../results/figures/{filename}", bbox_inches='tight', dpi=150)
    plt.show()
    print(f"Saved -> results/figures/{filename}")

save_degradation_curve("Accuracy",          "accuracy_vs_degradation.png")
save_degradation_curve("Macro F1",          "macro_f1_vs_degradation.png")
save_degradation_curve("High Risk Recall",  "high_risk_recall_vs_degradation.png")
save_degradation_curve("Brier Score",       "brier_score_vs_degradation.png", lower_better=True)


In [ ]:
# --- Performance Drop Figure ---
pivot = drop_df.pivot(index="Classifier", columns="Tier", values="Recall Drop")
pivot = pivot[[c for c in tier_order if c in pivot.columns]]

plt.figure(figsize=(10, 5))
pivot.plot(kind="bar", figsize=(12, 6), colormap="RdYlGn_r")
plt.axhline(y=HIGH_RISK_RECALL_DROP_THRESHOLD, color='red', linestyle='--',
            label=f"Safety threshold ({HIGH_RISK_RECALL_DROP_THRESHOLD:.0%} drop)")
plt.title("High Risk Recall Drop from Clean Baseline")
plt.ylabel("Recall Drop")
plt.xlabel("Classifier")
plt.xticks(rotation=15)
plt.legend()
plt.tight_layout()
plt.savefig("../results/figures/performance_drop.png", bbox_inches='tight', dpi=150)
plt.show()
print("Saved -> results/figures/performance_drop.png")


In [ ]:
# --- Degradation Validation Figure ---
# Requested vs actual missing rate per tier
val_summary = (
    cv_df.groupby("Tier")[["Missing_Rate_Target", "Actual_Missing_Rate"]]
    .mean()
    .reindex(tier_order)
)

fig, ax = plt.subplots(figsize=(9, 5))
x = range(len(val_summary))
width = 0.35
ax.bar([i - width/2 for i in x], val_summary["Missing_Rate_Target"],
       width, label="Target Missing Rate", color='steelblue')
ax.bar([i + width/2 for i in x], val_summary["Actual_Missing_Rate"],
       width, label="Actual Missing Rate", color='salmon')
ax.set_xticks(list(x))
ax.set_xticklabels(val_summary.index, rotation=15)
ax.set_ylabel("Missing Rate")
ax.set_title("Degradation Validation: Target vs Actual Missing Rate")
ax.legend()
plt.tight_layout()
plt.savefig("../results/figures/degradation_validation.png", bbox_inches='tight', dpi=150)
plt.show()
print("Saved -> results/figures/degradation_validation.png")


In [ ]:
# --- Performance Heatmap ---
mean_f1     = cv_df.groupby(["Classifier", "Tier"])["Macro F1"].mean().unstack()
mean_recall = cv_df.groupby(["Classifier", "Tier"])["High Risk Recall"].mean().unstack()

# Reorder columns
for df_ in [mean_f1, mean_recall]:
    cols = [c for c in tier_order if c in df_.columns]
    df_ = df_[cols]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.heatmap(mean_f1[tier_order], annot=True, fmt=".3f",
            cmap="YlGnBu", ax=axes[0])
axes[0].set_title("Mean Macro F1")

sns.heatmap(mean_recall[tier_order], annot=True, fmt=".3f",
            cmap="YlOrRd", ax=axes[1])
axes[1].set_title("Mean High Risk Recall")

plt.suptitle("Model Performance Heatmap by Degradation Tier", fontsize=13)
plt.tight_layout()
plt.savefig("../results/figures/performance_heatmap.png", bbox_inches='tight', dpi=150)
plt.show()
print("Saved -> results/figures/performance_heatmap.png")


In [ ]:
# --- Severe Condition Confusion Matrices (Tier 3 only) ---
classes = ['high risk', 'low risk', 'mid risk']

n_cls = len(severe_cms)
fig, axes = plt.subplots(1, n_cls, figsize=(5 * n_cls, 4))
if n_cls == 1:
    axes = [axes]

for ax, (name, cm) in zip(axes, severe_cms.items()):
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=classes, yticklabels=classes, ax=ax)
    ax.set_title(f"{name}\nTier 3 (Severe)")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

plt.suptitle("Confusion Matrices — Severe Degradation", fontsize=13)
plt.tight_layout()
plt.savefig("../results/figures/severe_confusion_matrix.png", bbox_inches='tight', dpi=150)
plt.show()
print("Saved -> results/figures/severe_confusion_matrix.png")


In [ ]:
print("=" * 50)
print("EXPERIMENT COMPLETE")
print("=" * 50)
print(f"  Folds:   {N_SPLITS}")
print(f"  Seeds:   {len(DEGRADATION_SEEDS)}")
print(f"  Tiers:   {list(TIERS.keys())}")
print(f"  Models:  {cv_df['Classifier'].unique().tolist()}")
print()
print("Saved outputs:")
print("  CSV    -> results/csv/robustness_results.csv")
print("  Tables -> results/tables/robustness_summary.csv")
print("          -> results/tables/model_comparison.csv")
print("          -> results/tables/degradation_summary.csv")
print("  Figures-> results/figures/accuracy_vs_degradation.png")
print("          -> results/figures/macro_f1_vs_degradation.png")
print("          -> results/figures/high_risk_recall_vs_degradation.png")
print("          -> results/figures/brier_score_vs_degradation.png")
print("          -> results/figures/performance_drop.png")
print("          -> results/figures/degradation_validation.png")
print("          -> results/figures/performance_heatmap.png")
print("          -> results/figures/severe_confusion_matrix.png")
